# Phase 5 & 6: Docker Containerisation & CI/CD

## Packaging the Full Stack for Production Deployment

**What this phase adds to the project:**

| Component | Technology | Purpose |
|---|---|---|
| **Full Stack Containers** | Docker Compose | One-command startup for DB + API + Dashboard |
| **API Container** | Docker | Isolated, reproducible API deployment |
| **Dashboard Container** | Docker | Isolated dashboard deployment |
| **CI Pipeline** | GitHub Actions | Automated linting and testing on every push |
| **Code Quality** | Ruff | Fast Python linter for consistent code style |

Containerisation and CI/CD make the project reproducible: `docker compose up` brings up the whole stack, and the CI workflow runs the linter and tests on every push.

---

## 0. Environment Setup (Google Colab)

In [ ]:
# ── Google Colab / Drive Setup ──
import sys
import os

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set project path — UPDATE THIS if your folder structure differs
PROJECT_PATH = "/content/drive/MyDrive/Github/Projects/saas-churn-prediction"

# Change to project directory
os.chdir(PROJECT_PATH)
sys.path.insert(0, PROJECT_PATH)

print(f"✓ Working directory: {os.getcwd()}")
print(f"✓ Project files: {os.listdir('.')[:10]}")

In [ ]:
# Install linting tools
!pip install ruff -q

print("✓ Setup complete")

---
## 1. Understanding the Docker Architecture

### Why Docker?

Docker solves the "it works on my machine" problem. By packaging code, dependencies, and configuration into containers, anyone can run the entire system — regardless of their operating system or installed software.

### Full Stack Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                    docker compose up                          │
│                                                              │
│  ┌─────────────┐   ┌──────────────┐   ┌─────────────────┐  │
│  │  PostgreSQL  │   │   FastAPI    │   │   Streamlit     │  │
│  │   Database   │◄──│     API      │   │   Dashboard     │  │
│  │  port 5432   │   │  port 8000   │   │   port 8501     │  │
│  └─────────────┘   └──────────────┘   └─────────────────┘  │
│                                                              │
│  Health checks ensure services start in the right order:     │
│  PostgreSQL → API (waits for DB) → Dashboard (waits for API) │
└──────────────────────────────────────────────────────────────┘
```

### What Each Dockerfile Does

**`api/Dockerfile`:**
- Starts from a slim Python 3.10 image
- Installs project + API dependencies
- Copies source code and model artifacts
- Runs uvicorn on port 8000
- Includes a health check endpoint

**`dashboard/Dockerfile`:**
- Starts from a slim Python 3.10 image
- Installs dashboard dependencies (Streamlit, Plotly)
- Copies dashboard code and data
- Runs Streamlit in headless mode on port 8501

---
## 2. Exploring the Docker Configuration

Let's examine each file to understand how the containers are configured.

In [ ]:
# ── View docker-compose.yml ──
print("═" * 60)
print("  docker-compose.yml")
print("═" * 60)
with open("docker-compose.yml") as f:
    print(f.read())

In [ ]:
# ── View API Dockerfile ──
print("═" * 60)
print("  api/Dockerfile")
print("═" * 60)
with open("api/Dockerfile") as f:
    print(f.read())

In [ ]:
# ── View Dashboard Dockerfile ──
print("═" * 60)
print("  dashboard/Dockerfile")
print("═" * 60)
with open("dashboard/Dockerfile") as f:
    print(f.read())

In [ ]:
# ── View .dockerignore ──
print("═" * 60)
print("  .dockerignore")
print("═" * 60)
with open(".dockerignore") as f:
    print(f.read())

---
## 3. CI/CD Pipeline with GitHub Actions

### What is CI/CD?

**Continuous Integration (CI)** means automatically running tests and checks every time code is pushed. This catches bugs early and ensures the codebase stays healthy.

### Our Pipeline

The GitHub Actions workflow (`.github/workflows/ci.yml`) runs on every push to `main` and every pull request:

1. **Checkout** — Pulls the code
2. **Setup Python** — Installs Python 3.10
3. **Install dependencies** — Installs all project requirements
4. **Lint with Ruff** — Checks code style and common errors
5. **Run tests** — Executes the 18 API tests
6. **Verify artifacts** — Confirms model files exist

### Why this matters

CI runs the linter and the test suite on a clean machine on every push, so a regression shows up immediately instead of being found later.

In [ ]:
# ── View the CI workflow ──
print("═" * 60)
print("  .github/workflows/ci.yml")
print("═" * 60)
with open(".github/workflows/ci.yml") as f:
    print(f.read())

---
## 4. Running the Linter

Let's run Ruff — the same linter that GitHub Actions will use — to check our code quality.

In [ ]:
# ── Run Ruff linter on the codebase ──
import subprocess

print("═" * 60)
print("  RUFF LINTING RESULTS")
print("═" * 60)

result = subprocess.run(
    ["python", "-m", "ruff", "check", "src/", "api/", "tests/"],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("\n✓ All checks passed — no issues found!")
else:
    print("\nIssues found:")
    print(result.stdout)
    print("\nRun 'ruff check --fix' to auto-fix where possible.")

---
## 5. Running the Test Suite

These are the same tests that GitHub Actions runs automatically.

In [ ]:
# ── Run pytest ──
!cd {PROJECT_PATH} && python -m pytest tests/test_api.py -v --tb=short 2>&1

---
## 6. How to Run the Full Stack Locally

While Docker can't run inside Google Colab, here's exactly how to start the full stack on your Windows laptop.

### Prerequisites

1. Install [Docker Desktop for Windows](https://www.docker.com/products/docker-desktop/)
2. Make sure Docker Desktop is running

### Start Everything

```bash
cd saas-churn-prediction
docker compose up --build
```

This will:
1. Pull the PostgreSQL 16 image
2. Build the API container from `api/Dockerfile`
3. Build the dashboard container from `dashboard/Dockerfile`
4. Start all three services with health checks

### Access the Services

| Service | URL | What You See |
|---|---|---|
| **API Docs** | http://localhost:8000/docs | Swagger UI — test predictions interactively |
| **Dashboard** | http://localhost:8501 | Full interactive churn risk dashboard |
| **Database** | localhost:5432 | PostgreSQL (connect via DBeaver, pgAdmin, etc.) |

### Stop Everything

```bash
docker compose down        # Stop containers
docker compose down -v     # Stop and delete database data
```

### Using Make (Optional)

```bash
make stack-up      # Start full stack
make stack-down    # Stop full stack
make stack-logs    # View logs
make test          # Run tests
make lint          # Run linter
```

In [ ]:
# ── Verify all Docker files are in place ──
from pathlib import Path

docker_files = {
    "docker-compose.yml": Path("docker-compose.yml"),
    "api/Dockerfile": Path("api/Dockerfile"),
    "dashboard/Dockerfile": Path("dashboard/Dockerfile"),
    ".dockerignore": Path(".dockerignore"),
    ".github/workflows/ci.yml": Path(".github/workflows/ci.yml"),
    "ruff.toml": Path("ruff.toml"),
    "Makefile": Path("Makefile"),
}

print("═" * 60)
print("  DEPLOYMENT FILES CHECK")
print("═" * 60)

all_present = True
for name, path in docker_files.items():
    if path.exists():
        size = path.stat().st_size
        print(f"  ✓ {name:35s} ({size:,} bytes)")
    else:
        print(f"  ✗ {name:35s} MISSING")
        all_present = False

print()
if all_present:
    print("✓ All deployment files present — ready for Docker and CI/CD!")
else:
    print("⚠ Some files are missing — check the file placement instructions.")

---
## 7. Adding the CI Badge to Your README

Once you push the `.github/workflows/ci.yml` file and the workflow runs successfully, GitHub will show a status badge. Add this to the **very top** of your `README.md`:

```markdown
![CI Pipeline](https://github.com/YOUR_USERNAME/saas-churn-prediction/actions/workflows/ci.yml/badge.svg)
```

Replace `YOUR_USERNAME` with your GitHub username.

The badge shows the current build status (passing or failing) directly in the README.

---

---
## Summary — Phases 5 & 6 Complete

### What was built

| Component | File | Purpose |
|---|---|---|
| **Docker Compose** | `docker-compose.yml` | Full stack orchestration (DB + API + Dashboard) |
| **API Dockerfile** | `api/Dockerfile` | Containerised prediction service |
| **Dashboard Dockerfile** | `dashboard/Dockerfile` | Containerised Streamlit dashboard |
| **Docker Ignore** | `.dockerignore` | Keep container images lean |
| **CI Pipeline** | `.github/workflows/ci.yml` | Automated linting and testing |
| **Linter Config** | `ruff.toml` | Consistent code style rules |
| **Makefile** | `Makefile` | Common commands for development |

### Technologies demonstrated

- **Docker** — Industry-standard containerisation
- **Docker Compose** — Multi-service orchestration with health checks
- **GitHub Actions** — CI/CD pipeline automation
- **Ruff** — Modern Python linter (10-100x faster than flake8)
- **Make** — Build automation (standard in engineering teams)



The SaaS Churn Prediction Platform is now a complete, end-to-end system:

| Phase | Component | Status |
|---|---|---|
| Phase 1 | Data Pipeline (SQL + Python) |  Complete |
| Phase 2 | ML Modelling (MLflow + SHAP) |  Complete |
| Phase 3 | FastAPI REST API |  Complete |
| Phase 4 | Streamlit Dashboard |  Complete |
| Phase 5 | Docker Containerisation |  Complete |
| Phase 6 | CI/CD with GitHub Actions | Complete |

End to end, the project covers the full lifecycle:
**data → features → model → API → dashboard → containers → CI/CD**